In [12]:
import pyodbc
import pandas as pd

# ============================================================
# 1. Cargar Excel (ya tiene la columna 'id')
# ============================================================
excel_path = "input/dbo_C_ccConvEscImpPorc_Soriana_20200720.xlsx"
df_excel = pd.read_excel(excel_path)

print("Excel cargado:", df_excel.shape)


# ============================================================
# 2. Preparar lista de IDs únicos para tabla Legado
# ============================================================
values = df_excel["Id"].dropna().astype(str).unique().tolist()

print("Valores únicos para consulta SQL:", len(values))


# ============================================================
# 3. Conexión a SQL Server
# ============================================================
conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=ATL20AF2222SQ19;"
    "DATABASE=SORIANA_MX_2024_PROD_F;"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(conn_str)


# ============================================================
# 4. Consulta por lotes a tabla LEGADO
# ============================================================
batch_size = 500
resultados_legado = []

tabla_legado = "dbo.Convenios_Legado_Soriana"

# ➤ REEMPLAZA ESTAS COLUMNAS POR LAS QUE EXISTEN EN TU TABLA
cols_legado = [
    "ID_NUM_CONV",
    "ID_NUM_VER",
    "ID_NUM_PLAZOPAGO",
    "DESC_CONVEVTO",
    "FECINI",
    "FECFIN",
    "ID_NUM_PROV",
    "DESC_CONVSTAT_NVO",
    "DESC_CONVEVTO_NVO",
]
cols_legado_select = ", ".join(cols_legado)

for i in range(0, len(values), batch_size):
    subset = values[i: i + batch_size]
    placeholders = ",".join("?" for _ in subset)

    query_legado = f"""
        SELECT 
            CONCAT(CAST(ID_NUM_CONV AS varchar(50)), CAST(ID_NUM_VER AS varchar(50))) AS id,
            {cols_legado_select}
        FROM {tabla_legado}
        WHERE CONCAT(CAST(ID_NUM_CONV AS varchar(50)), CAST(ID_NUM_VER AS varchar(50))) IN ({placeholders})
    """

    print(f"Batch LEGADO {i//batch_size + 1}")
    df_tmp = pd.read_sql(query_legado, conn, params=subset)
    resultados_legado.append(df_tmp)

# Convertir resultados LEGADO
df_sql_legado = pd.concat(resultados_legado, ignore_index=True) if resultados_legado else pd.DataFrame()

print("Filas recuperadas desde LEGADO:", len(df_sql_legado))


# ============================================================
# 5. Merge Excel + Legado
# ============================================================
df_excel["Id"] = df_excel["Id"].astype(str)
df_sql_legado["id"] = df_sql_legado["id"].astype(str)

df_merged = df_excel.merge(
    df_sql_legado,
    left_on="Id",   # columna del Excel
    right_on="id",  # columna generada en SQL
    how="left",
    suffixes=("", "_legado")
)

# Marcar origen = Legado
df_merged["origen"] = df_merged["ID_NUM_CONV"].notna().map({True: "Legado", False: None})

print("Filas con origen 'Legado':", df_merged["origen"].eq("Legado").sum())


# ============================================================
# 6. Filtrar filas SIN match en Legado
# ============================================================
df_sin_legado = df_merged[df_merged["origen"].isna()].copy()
print("Filas sin cruce en Legado:", df_sin_legado.shape[0])


# ============================================================
# 7. Consulta a tabla SAP KONA para esos faltantes
# ============================================================
faltantes_conv = df_sin_legado["Id_Num_Conv"].dropna().unique().tolist()

print("Registros para consultar en SAP_KONA:", len(faltantes_conv))

tabla_kona = "dbo.Catalogo_Convenios_SAP_KONA"

cols_kona = [
    "acuerdo",
    "clase_acuerdo",
    "descripción_del_acuerdo",
    "no_proveedor",
    "estatus",
    "fecha_modificación_cancelación"]

cols_kona_select = ", ".join(cols_kona)

query_kona = f"""
    SELECT {cols_kona_select}
    FROM {tabla_kona}
    WHERE acuerdo IN ({",".join("?" for _ in faltantes_conv)})
"""

df_sql_kona = pd.read_sql(query_kona, conn, params=faltantes_conv)

print("Filas recuperadas desde SAP_KONA:", len(df_sql_kona))


# ============================================================
# 8. Merge de los faltantes con SAP KONA
# ============================================================
df_sin_legado["Id_Num_Conv"] = df_sin_legado["Id_Num_Conv"].astype(str)
df_sql_kona["acuerdo"] = df_sql_kona["acuerdo"].astype(str)

df_kona_merge = df_sin_legado.merge(
    df_sql_kona,
    left_on="Id_Num_Conv",
    right_on="acuerdo",
    how="left",
    suffixes=("", "_kona")
)

df_kona_merge["origen"] = df_kona_merge["acuerdo"].notna().map({True: "Sap Kona", False: None})

print("Filas con origen 'Sap Kona':", df_kona_merge["origen"].eq("Sap Kona").sum())


# ============================================================
# 9. Unión final de resultados Legado + SAP_KONA
# ============================================================
df_legado_final = df_merged[df_merged["origen"] == "Legado"]
df_kona_final = df_kona_merge[df_kona_merge["origen"] == "Sap Kona"]

df_total = pd.concat([df_legado_final, df_kona_final], ignore_index=True)

print("FILAS TOTALES EN EL RESULTADO FINAL:", df_total.shape[0])


# ============================================================
# 10. Mostrar ejemplo final
# ============================================================
df_total.head()


Excel cargado: (5316, 8)
Valores únicos para consulta SQL: 1316
Batch LEGADO 1


C:\Users\opined01\AppData\Local\Temp\2\ipykernel_5024\3759620348.py:69: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tmp = pd.read_sql(query_legado, conn, params=subset)


Batch LEGADO 2
Batch LEGADO 3
Filas recuperadas desde LEGADO: 634
Filas con origen 'Legado': 2561
Filas sin cruce en Legado: 2771
Registros para consultar en SAP_KONA: 180


C:\Users\opined01\AppData\Local\Temp\2\ipykernel_5024\3759620348.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_kona = pd.read_sql(query_kona, conn, params=faltantes_conv)


Filas recuperadas desde SAP_KONA: 178
Filas con origen 'Sap Kona': 2164
FILAS TOTALES EN EL RESULTADO FINAL: 4725


,Id,Id_Num_Conv,Id_Num_Ver,Id_Cnsc_Esc,Imp_LimInf,Imp_LimSup,Num_Subtipo,Porc_Descontar,id,ID_NUM_CONV,...,ID_NUM_PROV,DESC_CONVSTAT_NVO,DESC_CONVEVTO_NVO,origen,acuerdo,clase_acuerdo,descripción_del_acuerdo,no_proveedor,estatus,fecha_modificación_cancelación
0,10487,1048,7,1,0.00,8.500000e+05,1,0.0,10487,1048.0,...,57265.0,Cancelado,Cancelación,Legado,NaN,NaN,NaN,NaN,NaN,NaN
1,10487,1048,7,2,850000.00,2.500000e+06,1,1.5,10487,1048.0,...,57265.0,Cancelado,Cancelación,Legado,NaN,NaN,NaN,NaN,NaN,NaN
2,10487,1048,7,3,2500000.01,4.500000e+06,1,1.6,10487,1048.0,...,57265.0,Cancelado,Cancelación,Legado,NaN,NaN,NaN,NaN,NaN,NaN
3,10487,1048,7,4,4500000.01,6.500000e+06,1,1.7,10487,1048.0,...,57265.0,Cancelado,Cancelación,Legado,NaN,NaN,NaN,NaN,NaN,NaN
4,10487,1048,7,5,6500000.01,1.000000e+09,1,1.8,10487,1048.0,...,57265.0,Cancelado,Cancelación,Legado,NaN,NaN,NaN,NaN,NaN,NaN
